# Lab 03 – Network traffic and its effect on QoE

Packets are crafted and sent with Scapy, first a text message, then a speech recording as G.711 in RTP. The final section scores the recordings received under emulated network impairments with PESQ and ViSQOL and relates them to the subjective ratings. Theory, the `tc`, tcpreplay, and tshark procedures, and the tool builds are in the [README](../README.md).

Before starting, capture a call to the laboratory PBX (README, Step 1) and prepare `input.wav`: mono, 8 kHz, signed 16-bit PCM. Sending and sniffing packets requires root privileges or the `cap_net_raw` capability on the interpreter (README, Preparation).

## Imports

In [ ]:
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scapy.all as sa
from IPython.display import Markdown, display
from scipy.io import wavfile

from lib.core import linear_to_alaw, linear_to_ulaw, score_pair
from lib.params import add_logo, colors, rc_params

plt.rcParams.update(rc_params)

## Constants

In [ ]:
rng = np.random.default_rng()

SRC_IP = "127.0.0.1"  # own address, see `ip a`
DST_IP = "127.0.0.1"  # address of the receiving colleague
SRC_PORT = 12345
DST_PORT = 54321
IFACE = "lo"  # interface to sniff on

SAMPLES_PER_PACKET = 160  # 20 ms of G.711 at 8 kHz

# Quality evaluation (Step 3)
RESULTS_DIR = Path("results")
REFERENCE = RESULTS_DIR / "reference.wav"
PESQ_BIN = Path.home() / "PESQ" / "PESQ"  # README, Step 5
VISQOL_BIN = Path.home() / "visqol" / "bazel-bin" / "visqol"  # README, Step 6
FILE_PATTERN = re.compile(r"output_(?P<model>random|gemodel)_(?P<loss>[\d.]+)\.wav")

## Theory

### Packet crafting with Scapy

Scapy represents each protocol as a layer and composes packets with the `/` operator. A layer is created with the fields that matter; Scapy fills in the rest (lengths, checksums) when the packet is finalized for sending. The following cells build an ICMP echo request carrying a text message and show it at each stage.

In [ ]:
ip = sa.IP(src=SRC_IP, dst=DST_IP)
ip

Fields as set so far; unset fields show their defaults.

In [ ]:
ip.show()

The same layer after finalization, that is, as it will appear on the wire.

In [ ]:
ip.show2()

An [ICMP](https://en.wikipedia.org/wiki/Internet_Control_Message_Protocol) echo request (type 8) is stacked on top of the IP layer, and the message becomes its payload.

In [ ]:
icmp = sa.ICMP(type=8)
msg = "Hello world"

packet = ip / icmp / msg
packet.show2()

In [ ]:
sa.hexdump(packet)

#### Question

Where else have you seen this representation of a packet, and what do the columns mean?

## Tasks

### Step 1 – Text exchange

The sender transmits the packet; the receiver sniffs the interface and prints the payload of every echo request. Run the receiver in a second notebook or on the colleague's machine, since `sniff()` blocks the kernel until interrupted.

In [ ]:
sa.send(packet)

In [ ]:
def print_payload(pkt):
    if pkt["ICMP"].type == 8:
        print(pkt["Raw"].load.decode(errors="replace"), end="")


sa.sniff(iface=IFACE, prn=print_payload, filter="icmp", store=0)

### Task 1 – UDP, one character per packet

1. Rewrite the sender to use UDP (`sa.UDP(sport=SRC_PORT, dport=DST_PORT)`) and to send the message one character per packet.
2. Adapt the receiver's filter and callback accordingly.
3. Explain what changes on the wire between the ICMP and the UDP version.

### Task 2 – Random delay and loss

1. Add a random inter-packet delay drawn from `rng` to the sender.
2. Add random packet loss with a configurable probability.
3. Capture the result with Wireshark and confirm both effects from the timestamps and the missing characters.

### Step 2 – Audio over RTP

The function reads `input.wav`, converts each block of 160 samples to G.711 with the companding functions from `lib/core.py`, wraps it in an RTP header, and sends it as UDP.

In [ ]:
def send_audio_as_rtp(path="input.wav", encoding="alaw"):
    """
    Send a WAV file (8 kHz, mono, int16) as G.711 RTP packets to DST_IP:DST_PORT.

    Parameters
    ----------
    path : str
        Input WAV file.
    encoding : {"alaw", "ulaw"}
        G.711 variant: A-law (payload type 8) or μ-law (payload type 0).
    """
    if encoding == "ulaw":
        payload_type, converter = 0, linear_to_ulaw
    elif encoding == "alaw":
        payload_type, converter = 8, linear_to_alaw
    else:
        raise ValueError("encoding must be 'ulaw' or 'alaw'")

    sa.bind_layers(sa.UDP, sa.RTP, dport=DST_PORT)

    sample_rate, samples = wavfile.read(path)
    if sample_rate != 8000 or samples.dtype != np.int16:
        raise ValueError("audio must be 8000 Hz, 16-bit PCM")

    sequence_number = 0
    timestamp = 0
    for i in range(0, len(samples), SAMPLES_PER_PACKET):
        chunk = samples[i : i + SAMPLES_PER_PACKET]
        if len(chunk) < SAMPLES_PER_PACKET:
            chunk = np.pad(chunk, (0, SAMPLES_PER_PACKET - len(chunk)))

        packet = (
            sa.IP(dst=DST_IP)
            / sa.UDP(sport=SRC_PORT, dport=DST_PORT)
            / sa.RTP(
                version=2,
                padding=0,
                extension=0,
                marker=0,
                payload_type=payload_type,
                sequence=sequence_number,
                timestamp=timestamp,
                sourcesync=0x12345678,
            )
            / sa.Raw(load=bytes(converter(chunk)))
        )
        sa.send(packet, verbose=False)

        sequence_number = (sequence_number + 1) & 0xFFFF
        timestamp = (timestamp + SAMPLES_PER_PACKET) & 0xFFFFFFFF
        time.sleep(0.01)  # nominal packetization interval is 20 ms; see Task 3

In [ ]:
send_audio_as_rtp("input.wav")

### Task 3 – Assess the audio transmission

1. Capture the stream with `tcpdump` or Wireshark, decode it as RTP, and play it (*Telephony → RTP → RTP Player*).
2. From the capture, assess the packet spacing the sender achieves and compare it with the nominal 20 ms. Which parts of the function limit the timing precision?
3. Apply delay and loss with `tc netem` (README, Step 3) and repeat the transmission.
4. Plot the inter-packet intervals of the received stream against the packet sequence number, and mark lost packets.

Continue with the README, Steps 4 to 6, to produce the degraded recordings evaluated below.

### Step 3 – Quality evaluation

This section brings the measurements of the multi-week activity together: the loss and jitter read from Wireshark, the subjective ACR and DCR ratings, and the objective PESQ and ViSQOL scores computed here.

Place the recordings in a `results/` directory next to this notebook, named after the impairment that produced them:

```
results/
├── reference.wav             clean reference (8 kHz, mono, 16-bit)
├── output_random_0.5.wav     netem loss 0.5%
├── output_random_1.0.wav
├── output_random_3.0.wav
├── output_gemodel_1.0.wav    netem loss gemodel ...
└── ...
```

The pattern is `output_<model>_<loss>.wav`, where `<model>` is `random` or `gemodel` and `<loss>` the configured average loss in percent.

`score_pair()` in `lib/core.py` calls the PESQ and ViSQOL executables; their locations are set in the Constants cell. A missing tool yields `nan` rather than an error, so the table can be filled in stages.

Reconstruct each WAV with the timeline-preserving utility in README Step 5. Missing intervals contain silence; all recordings retain the clean stream duration. Keep repeated trials in separate directories and update RESULTS_DIR and REFERENCE together. Retain individual listener ratings separately.

#### Measured and subjective values

Values that are not produced by code are entered by hand: the loss and jitter reported by Wireshark for each received stream (*Telephony → RTP → RTP Streams*) and the ACR and DCR ratings averaged over the group. Running the next cell prints a template with one entry per recording found in `results/`; copy it into the cell below and fill in the numbers.

Use the separate ACR and DCR anchors in the README. Record the listener count and spread of ratings. ACR MOS and DCR DMOS have different meanings despite both using five categories.

In [ ]:
recordings = sorted(
    f for f in RESULTS_DIR.glob("output_*.wav") if FILE_PATTERN.fullmatch(f.name)
)

print("manual = {")
for f in recordings:
    print(f'    "{f.name}": dict(measured_loss=None, jitter=None, acr=None, dcr=None),')
print("}")

In [ ]:
# Measured loss [%], jitter [ms], ACR MOS and DCR DMOS [–]
manual = {
    # "output_random_0.5.wav": dict(measured_loss=0.6, jitter=2.1, acr=4.2, dcr=4.0),
}

#### Objective scores

Each recording is scored against the reference. PESQ reads the 8 kHz files directly; ViSQOL is run in speech mode on copies resampled to 16 kHz.

In [ ]:
rows = []
for f in recordings:
    m = FILE_PATTERN.fullmatch(f.name)
    entry = manual.get(f.name, {})
    rows.append(
        dict(
            file=f.name,
            model=m["model"],
            loss=float(m["loss"]),
            measured_loss=entry.get("measured_loss"),
            jitter=entry.get("jitter"),
            acr=entry.get("acr"),
            dcr=entry.get("dcr"),
            **score_pair(REFERENCE, f, pesq_bin=PESQ_BIN, visqol_bin=VISQOL_BIN),
        )
    )

rows.sort(key=lambda r: (r["model"], r["loss"]))


def fmt(v):
    return "–" if v is None or v != v else f"{v:.2f}"


header = "| Loss model | Loss set [%] | Loss measured [%] | Jitter [ms] | ACR MOS [–] | DCR DMOS [–] | PESQ estimate* [–] | ViSQOL MOS-LQO [–] |"
lines = [header, "|" + "---|" * 8]
for r in rows:
    lines.append(
        f"| {r['model']} | {r['loss']:.1f} | {fmt(r['measured_loss'])} | {fmt(r['jitter'])} "
        f"| {fmt(r['acr'])} | {fmt(r['dcr'])} | {fmt(r['pesq'])} | {fmt(r['visqol'])} |"
    )
display(Markdown("\n".join(lines)))
display(
    Markdown(
        "*The PESQ helper uses MOS-LQO when present, otherwise the raw prediction. Check the executable output and identify raw results separately."
    )
)

#### Quality scores against packet loss

One line per method; solid lines and circles are random loss, dashed lines and squares the Gilbert–Elliott model.

Compare trends and rankings; equal scores from different methods do not imply equal judgments. The horizontal axis gives configured mean loss; compare it with measured loss in the table.

In [ ]:
METHODS = [
    ("acr", "ACR MOS", colors[0]),
    ("dcr", "DCR DMOS", colors[4]),
    ("pesq", "PESQ", colors[5]),
    ("visqol", "ViSQOL", colors[6]),
]
MODELS = [("random", "-", "o", "random"), ("gemodel", "--", "s", "Gilbert–Elliott")]

fig, ax = plt.subplots(figsize=(10, 6))
add_logo(fig)

for key, label, color in METHODS:
    for model, ls, marker, model_label in MODELS:
        pts = [
            (r["loss"], r[key])
            for r in rows
            if r["model"] == model and r[key] is not None and r[key] == r[key]
        ]
        if pts:
            x, y = zip(*pts)
            ax.plot(
                x, y, ls, marker=marker, color=color, label=f"{label}, {model_label}"
            )

ax.set_xlabel("Configured mean packet loss [%]")
ax.set_ylabel("Quality score [–]")
# Allow raw PESQ predictions outside the mapped 1–5 scale to remain visible.
ax.set_title("Speech quality against packet loss")
ax.legend(loc="lower left")
plt.show()

### Task 4 – Compare the methods

- Complete the table for at least three loss rates of each loss model.
- Where do PESQ and ViSQOL disagree in their ranking of conditions, and which of them follows the ACR and DCR ratings more closely?
- Does the same average loss produce similar ratings under both loss models? Explain the difference using the burst length of the Gilbert–Elliott model.
- Which columns of the table would change in a conversational test, and why do the objective scores not capture it?

## Questions

1. Which layer of the TCP/IP stack does each of SIP, RTP, UDP, and IP belong to, and which of them carries the codec information?
2. What is the IP-level bit rate of a G.711 call in both directions, and how does it change if the packetization interval is 40 ms?
3. How closely did the observed packet spacing match 20 ms? Which software and operating-system effects can introduce timing variation?
4. Two links have the same 3 % average loss, one independent and one bursty. Predict how speech quality might differ, then explain how speech content and packet-loss concealment could affect the result.
5. Which of your measurements would change between a listening test and a conversational test, and why do PESQ and ViSQOL ignore it?
6. PESQ and ViSQOL are both intrusive models. What does a *non-intrusive* model have to do differently, and where in a VoIP system would it be needed?